## Environment Setup & Imports

If you are running this notebook on **Databricks**, uncomment and run the following line in the cell below to install the local `landseg` package in your cluster's Python environment:
```python
# %pip install -e ..
```

In [2]:
# Databricks Setup

import os
import sys
sys.path.insert(0, os.path.join(os.path.abspath('..'), 'src'))
import landseg.adapters.api as api

# Check if running in a Databricks environment
IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

# Define the experiment root path where artifacts and results will be stored.
# Relative paths are standard for local repositories. On Databricks, consider utilizing
# absolute paths on a Unity Catalog Volume or DBFS path (e.g. '/Volumes/my_catalog/my_schema/my_volume/experiment/').
EXP_ROOT = '../experiment/'
# Define name of the dataset, used for naming outputs and logging
DATASET_NAME = 'demo_data'
#
if IS_DATABRICKS:
    # EXP_ROOT = '/Volumes/my_catalog/my_schema/my_volume/experiment/'
    print("Running on Databricks. For high performance, configure EXP_ROOT using a Unity Catalog Volume or DBFS path.")

## Configure Data Ingestion and Run

In [2]:
# NOTE
# This is a demo script to how to configure the data ingestion pipeline
# The purpose is to process raw rasters into stable artifacts (.npz files) mapped to the grid
# The data artifacts are cataloged for access during data preparation; training will not read them
# Typically this is run once unless input data is changed or you want to adjust the settings
# Adjust the parameters and file paths as needed for your specific use case
# Relative file paths are used here for demonstration (data shipped with the codebase)
# Use absolute file paths to ensure consistency across different environments

# init configurator
configurator = api.DataIngestionConfigurator(EXP_ROOT, DATASET_NAME)

# set whether to force rebuild ingestion data artifacts (default: False)
configurator.set_rebuild(False)

# set grid parameters for tiling the input data
configurator.set_grid(
    # EPSG code for the grid reference raster CRS
    crs='EPSG:2958',
    # path to the reference raster used for defining the tiling grid and extent
    reference_raster_fpath='../experiment/input/extent_reference/extent_reference.tif',
    # size of the tiles to be generated in pixels - here we supoort only square tiles
    tile_size=256,
    # used for data augmentation and to avoid edge effects during training
    tile_overlap=128
)

# set domain knowledge layers to be used as additional input channels for the model
configurator.set_domains([
    # each entry should contain the file path and the raster index base
    # ('../experiment/input/domain_knowledge/....tif', 1),
    # ('../experiment/input/domain_knowledge/....tif', 1)
])

# set training data for model development (i.e., training and validation)
configurator.set_model_dev_data(
    model_dev_image='../experiment/input/data/demo_data/dev_image.tif',
    model_dev_label='../experiment/input/data/demo_data/dev_label.tif',
    data_config='../experiment/input/data/demo_data/config.json',
)

# set test holdout data for final evaluation of the trained model
configurator.set_test_holdout_data(
    # optional - if not provided, model evaluation pipeline will fail
    test_holdout_image='../experiment/input/data/demo_data/test_image.tif',
    test_holdout_label='../experiment/input/data/demo_data/test_label.tif',
)

# run data ingestion
api.run(configurator.running_root_config)

2026-07-15 15:20:38,524-api-INFO	- Running pipeline: data-ingest
2026-07-15 15:21:14,527-ingest-INFO	- ==========================================================================================
2026-07-15 15:21:14,554-ingest-INFO	- [START] World grid preparation
2026-07-15 15:21:19,111-ingest-INFO	- [COMPLETE] World grid preparation (D_0.40s)
2026-07-15 15:21:19,112-ingest-INFO	- [NOTE] No domain knowledge layers provided
2026-07-15 15:21:19,112-ingest-INFO	- [START] Development data blocks building
 - Checking datablocks: 100%|████████████████████████████████████| 400/400 [00:02<00:00, 167.71it/s]
2026-07-15 15:21:23,438-ingest-INFO	- [COMPLETE] Development data blocks preparation (D_4.27s)
2026-07-15 15:21:23,439-ingest-INFO	- [START] Test data blocks building
 - Checking datablocks: 100%|███████████████████████████████████████| 36/36 [00:00<00:00, 49.08it/s]
2026-07-15 15:21:25,221-ingest-INFO	- [COMPLETE] Test data blocks preparation (D_1.78s)
2026-07-15 15:21:25,272-ingest-INFO	- 

## Configure Data Preparation and Run

In [5]:
# NOTE
# Data preparation is the intermediate step between data ingestion and model training
# Main processes include:
# 1. partitioning the model development data into training and validation sets
# 2. performing data augmentation if configured
# 3. normalize data and generate training-scoped artifacts (.npz files) and data schema artifacts
# Once this step is run, the training pipeline can be run directly from the artifacts
# Re-run if you want to adjust how the training view the data at runtime

# init configurator
configurator = api.DataPreparationConfigurator(EXP_ROOT, DATASET_NAME)

# set whether to force rebuild preparation data artifacts (default: False)
configurator.set_rebuild(False)

# set partitioning strategy for model development data
# partitioning will be done on the tiles from non-overlapping grid blocks
configurator.set_partition(
    # percentage of dev tiles to be used for validation, rest will be used for training
    validation_blocks_ratio=0.15,
    # percentage of dev tiles to be held out for testing
    # if test holdout data is provided, this parameter will be ignored and the holdout data will be used for testing
    test_holdout_blocks_ratio=0.1
)

# set data augmentation and sampling strategy for oversampling under-represented classes in the training data
# augemntation will source from overlapping tiles and will be skipped if tile_overlap is set to 0 in the grid configuration
# set target_head to None or leave reward_classes dictionary empty for no oversampling
configurator.set_oversampling(
    target_head=None,
    # class ID: score value for oversampling (higher value means more oversampling)
    # make sure the class ID matches the label values in the training data
    reward_classes={}
)

# run data preparation
api.run(configurator.running_root_config)

2026-07-15 15:30:08,555-api-INFO	- Running pipeline: data-prepare
2026-07-15 15:30:08,558-prep-INFO	- ==========================================================================================
2026-07-15 15:30:08,576-prep-INFO	- [START] Dataset partitioning splits
2026-07-15 15:30:08,580-prep-INFO	- [CHECKPOINT] Loaded dataset partition splits
2026-07-15 15:30:08,580-prep-ERROR	- Preparation pipeline failed: 
Traceback (most recent call last):
  File "c:\Users\FengWen\Documents_Local\caribou_landcover\src\landseg\execution\pipelines\data_prepare.py", line 100, in prepare
    transform.run_datablocks_partition(
  File "c:\Users\FengWen\Documents_Local\caribou_landcover\src\landseg\geopipe\transform\data_partition\runner.py", line 105, in run_datablocks_partition
    assert report # typing
           ^^^^^^
AssertionError
2026-07-15 15:30:08,582-prep-INFO	- ==========================================================================================
2026-07-15 15:30:08,600-api-CRITICAL	- Un

AssertionError: 